# Trustworthy AI begins with a risk model

**Core time:** 100 minutes. **Prerequisites:** ML evaluation and probability. **Outcomes:** distinguish reliability, robustness, privacy and fairness questions; specify a threat model; separate distribution shift from intentional manipulation; design a bounded audit.

## 1. A trustworthy system requires more than a good average score

A model can have strong held-out accuracy and still fail when inputs shift, expose sensitive information, perform unevenly across groups or be used outside its intended purpose. These are different problems with different evidence requirements. A robustness test does not establish privacy. A fairness metric does not establish causal fairness. Encryption during transport does not establish that model outputs reveal nothing about training examples.

The NIST AI RMF 1.0 provides a risk-management structure around Govern, Map, Measure and Manage. We use it as a framework for asking who is responsible, what context the system operates in, how risks are measured and what actions follow. This course is not a compliance certification, and completing a toy benchmark does not satisfy every organizational or domain-specific requirement.

## 2. Write the threat model before selecting a method

A threat model identifies the asset, adversary goal, knowledge, access, capabilities and constraints. For a teaching classifier, the asset might be reliable prediction, the goal untargeted misclassification, knowledge full model parameters, access inference inputs and constraint an L-infinity perturbation no larger than 0.1 in a `[0,1]` input domain. State whether the adversary acts during training or inference and whether success is targeted to a chosen label.

An alternative training-time threat allows modifying 10% of training examples with a visible trigger and changing their labels to zero. That is a different experiment from inference-time perturbation. Reporting only “attack success” without the allowed actions makes comparisons meaningless. The attacker budget, eligible population and defender knowledge must be explicit.

| Field | Example for a local teaching experiment |
|---|---|
| System and data | Own small classifier; bundled or synthetic data |
| Asset | Correct predictions on the stated input population |
| Goal | Untargeted error, or a separately defined target label |
| Access | Input gradients and model weights in white-box evaluation |
| Budget | Explicit norm/range or poisoning fraction |
| Defender | Frozen baseline or specified retrained variant |
| Success measure | Defined numerator, denominator and clean performance |
| Scope | Local educational model; no third-party service or private data |

## 3. Distribution shift is not one phenomenon

Covariate shift changes the input distribution under a particular assumption about the conditional label relationship. Label shift changes class prevalence under a different conditional invariance assumption. Concept shift changes the relationship between input and label. Real deployments may combine several changes. A corruption test such as added Gaussian noise is a controlled perturbation, not evidence that every real-world shift is covered.

Worked example: a digit model trained on centered images is evaluated on images shifted one pixel right. Accuracy drops. This observation supports a claim about that transformation and dataset. It does not prove malicious manipulation, nor does it imply that retraining on shifted examples solves all domain shift. Record the transformation, padding rule and whether labels are plausibly preserved; on an 8-by-8 image, a large transformation may erase the digit.

## 4. Construct a risk register with actions

A useful risk entry connects a failure mechanism to affected stakeholders, evidence and a decision. For example: “High-confidence mistakes under modest brightness change; measured on a held-out corruption set; investigate training augmentation and confidence calibration; do not extend the model to unseen acquisition conditions on this evidence alone.” The entry should name an owner and an action, not merely a severity adjective.



In [ ]:
risk = {
    'failure': 'prediction changes under a specified brightness shift',
    'population': 'held-out bundled digits',
    'metric': 'accuracy by predeclared shift magnitude',
    'control': 'identical unmodified held-out examples',
    'action': 'investigate preprocessing and document scope',
    'owner': 'student researcher and assigned reviewer'
}
required = {'failure','population','metric','control','action','owner'}
assert required <= risk.keys()



The dictionary checks documentation completeness, not safety. An entry can contain every field and still be scientifically wrong. Review whether the control, metric and decision actually relate to the stated risk.

## 5. Build an audit that can fail

Choose one primary risk question, define the expected failure condition and preserve a clean baseline. Evaluate a fixed stress grid and show all settings. If a method improves one metric but worsens another, report both. A model card should identify intended use, excluded use, data, evaluation, limitations and unresolved risks. Never infer absence of risk from an experiment that had no power to reveal it.

## Exercises and self-check

1. Write distinct threat models for test-time perturbation and training-time poisoning. **Check:** timing, permitted modifications, knowledge and goals differ.
2. A model has 99% clean accuracy. What additional information is needed for a robustness claim? **Check:** threat set, attack procedure, budget, denominators and scope.
3. Propose one non-adversarial shift test and explain how you would assess label preservation.
4. Convert “privacy risk is low” into an auditable statement. **Check:** specify the information, observer, access, mechanism and evidence; a vague adjective is insufficient.

## Primary reading

- [NIST: AI RMF 1.0](https://www.nist.gov/publications/artificial-intelligence-risk-management-framework-ai-rmf-10): use the four functions to organize responsibility and evidence.
- [Mitchell et al.: Model Cards for Model Reporting](https://arxiv.org/abs/1810.03993): adapt the reporting questions to the small model you actually evaluated.


# Adversarial robustness: gradients, constraints and measured failure

**Core time:** 2–3 hours. **Prerequisites:** gradients, norm budgets and a trained classifier. **Outcomes:** derive an FGSM step; implement projected iterative evaluation on your own model; verify perturbation constraints; distinguish empirical attack accuracy from certified robustness.

## 1. Turn the threat model into an optimization problem

For an untargeted inference-time attack, seek a permitted perturbation that increases the loss on the true label. The model is frozen. In this course the input domain is `[0,1]` and the permitted L-infinity displacement is at most epsilon. Both constraints matter: a point inside the norm ball can still be outside the legal input range.

```math
\max_{\delta}\;\ell(f_\theta(x+\delta),y)
\quad\text{subject to}\quad\|\delta\|_\infty\le\epsilon,\;x+\delta\in[0,1]^d.
```

The linear approximation to loss is `loss(x)+gradient.T@delta`. Under an L-infinity budget, each coordinate maximizes its contribution by taking epsilon times the gradient sign. This gives the fast gradient sign method. It is a local first-order construction; it does not solve every nonlinear constrained maximization exactly.

## 2. Worked example: a bounded update

For `x=[0.2,0.9]`, input gradient `[3,-2]` and epsilon 0.15, the sign step gives `[0.35,0.75]`, within the legal range and exactly 0.15 away in each changed coordinate. If the first input were 0.95 and its gradient positive, clipping would produce 1.0, a displacement of only 0.05. The actual perturbation norm can be smaller than the requested maximum because of domain boundaries.



In [ ]:
import torch
x = torch.tensor([[.2,.9]])
gradient = torch.tensor([[3.,-2.]])
epsilon = .15
adv = (x + epsilon*gradient.sign()).clamp(0,1)
assert (adv-x).abs().max() <= epsilon+1e-7
torch.testing.assert_close(adv,torch.tensor([[.35,.75]]))
print(adv)



For a real model, create an input tensor requiring gradients, compute loss and request its input gradient. Keep the model in evaluation mode. Do not wrap the attack construction in `no_grad`, and do not update model parameters during evaluation.

## 3. Iteration needs projection and a fixed reference input

Projected gradient descent repeatedly takes a gradient-ascent step, then clips the candidate to the original input's epsilon box and legal range. Projecting around the previous iterate instead would allow the total displacement to grow across steps. A random start samples a permitted initial point; multiple restarts can find different failures. Record step count, step size, restart count, random seed and whether the attack is targeted.

The bundled robustness lab implements these checks explicitly on the learner's own digit classifier. It includes the clean input as a candidate and tracks per-example failure across candidates. This is important: selecting a high-loss candidate alone is not identical to selecting any successful misclassification. Report the actual success rule.

## 4. Metrics need an eligible population

Clean accuracy counts correct unmodified predictions over all test examples. Accuracy on one attack's returned adversarial examples is an empirical measurement for that procedure. Worst-case robust accuracy over an entire allowed set is generally lower or equal to the accuracy under a particular attack, because the attack may miss a failing point. An empirical attack therefore does not certify robustness.

For attack success conditioned on initially correct examples, divide by the number of clean-correct examples. If this denominator is zero, report undefined. For an overall robust-correct indicator, require clean correctness and survival of every tested attack candidate. Show both the original test count and the conditioned denominator, so clean errors are not accidentally hidden.

## 5. Failure checks before interpreting a curve

At epsilon zero, the attacked input should equal the original. Increasing the tested candidate set cannot improve a per-example worst-case survival indicator when previous candidates are retained. However, two independently optimized attacks at different epsilons can produce nonmonotonic empirical curves because optimization is imperfect. Do not smooth away this evidence; investigate attack strength, random starts and implementation.

Gradient masking can make a gradient-based attack look weak without making a defense robust. A nondifferentiable or stochastic defense requires an evaluation suited to that behavior; a failed naive attack is not proof. In this introductory lab we use a simple differentiable model and avoid claiming an exhaustive defense evaluation.

## Guided lab and exercises

Run `python lab04_trust.py --output outputs/trust`. Inspect clean accuracy, epsilon-zero identity, L-infinity bounds and conditioned success counts. Then:

1. Change the input scaling from `[0,1]` to `[0,16]` and explain how the same physical perturbation budget should change. **Check:** multiply the numeric budget by 16; otherwise the threat changes.
2. Explain the sign reversal for a targeted loss minimizing the target-class cross-entropy. **Check:** targeted optimization uses descent on target loss, unlike untargeted ascent on true-label loss.
3. Compare one-step and iterative procedures under the same allowed set. Report stronger failure if found, not a universal ranking.
4. Why is “90% accuracy after FGSM” not “90% certified robust accuracy”? **Check:** FGSM explores only a particular construction inside the full threat set.

## Primary reading

- [Goodfellow, Shlens and Szegedy: Explaining and Harnessing Adversarial Examples](https://arxiv.org/abs/1412.6572): read the linearized sign-step argument.
- [Madry et al.: Towards Deep Learning Models Resistant to Adversarial Attacks](https://arxiv.org/abs/1706.06083): identify the constrained inner maximization and the empirical evaluation assumptions.
- [PyTorch: FGSM tutorial](https://docs.pytorch.org/tutorials/beginner/fgsm_tutorial.html): inspect input-gradient handling, while keeping this course's denominators explicit.


# A controlled backdoor study on a local teaching model

**Core time:** 2–3 hours. **Prerequisites:** training/evaluation separation and threat models. **Outcomes:** distinguish backdoors from ordinary test-time perturbations; construct a bounded training-only poisoning experiment; report clean and triggered metrics with correct eligibility; test a meaningful negative control.

## 1. Define the phenomenon without hiding the assumptions

A backdoored model can retain useful behavior on ordinary inputs while responding to a particular trigger in an attacker-chosen way. In a simple teaching threat model, the attacker can modify a small fraction of training examples and their labels. During evaluation, a visible trigger is applied to held-out inputs and the model is checked for prediction of a fixed target class. This is not the same as finding a small gradient perturbation after training.

Our lab operates only on a locally trained digit classifier and bundled data. The trigger is an obvious 2-by-2 bright patch in the bottom-right corner; the target class is zero; the poisoning fraction is explicitly recorded. It is a demonstration of a mechanism, not a reproduction of the original BadNets experimental numbers and not a test of any third-party system.

## 2. Preserve a clean baseline

Train a clean model and a poisoned model with matched architecture, initialization seed, split and training policy. Modify only the poisoned model's training copy. Never mutate the shared source array in place. Validation and test targets stay unchanged. Record which training indices were selected, the eligible selection pool, the number changed and whether the poisoning percentage is defined relative to all training examples or only non-target examples.

The teaching lab chooses approximately 10% of the total training count from eligible non-target training examples, applies the patch and assigns target zero. The exact integer count is saved. This explicit convention prevents a denominator mismatch when comparing with another paper or implementation.



In [ ]:
import numpy as np
def add_trigger(flat_images):
    out = np.array(flat_images,copy=True).reshape(-1,8,8)
    out[:,-2:,-2:] = 1.0
    return out.reshape(-1,64)
original = np.zeros((3,64),dtype=np.float32)
triggered = add_trigger(original)
assert np.count_nonzero(original) == 0
assert np.count_nonzero(triggered) == 3*4
assert triggered.min() >= 0 and triggered.max() <= 1



The patch may obscure meaningful pixels in a tiny image. That is a limitation to examine, not an assumption to ignore. Show representative clean/triggered pairs and avoid claiming that the trigger is imperceptible.

## 3. Worked example: eligibility changes the claimed success

Suppose there are 100 held-out examples and 10 already belong to target class zero. Among the 90 non-target examples after triggering, 72 are predicted as zero. The non-target attack success rate is `72/90=80%`. Including original target examples can inflate the statistic because predicting their original class is not an attacker-induced change to the target.

Also report clean accuracy of the poisoned model. A model that predicts zero for every input can achieve high target success while destroying normal utility. Triggered accuracy against the original labels measures a different quantity. If success is additionally conditioned on the model being clean-correct, state that denominator separately and do not compare it directly with an unconditioned ASR.

## 4. Negative controls separate the trigger from ordinary corruption

Apply the same trigger to the clean model. If its target rate also rises strongly, part of the effect may come from the trigger naturally resembling a class or obscuring evidence. Compare clean-model triggered behavior with poisoned-model triggered behavior. A zero-poisoning setting is another control. A patch at a different location can probe specificity, but changing location also changes the affected image information; interpret carefully.

Run several poisoning fractions chosen before evaluating test. Use validation data for method development and reserve test for frozen settings. With a small dataset, variation across poisoning selections and training seeds can be substantial. Preserve the selected indices and report per-run values instead of asserting that one convenient run defines the phenomenon.

## 5. What a defense claim would require

A proposed defense should be evaluated for clean utility, triggered success, knowledge assumptions and resource costs. If it uses a known trigger to filter examples, describe that knowledge; it is not a general unknown-trigger detector. If it removes suspicious data, report how many clean examples are removed and whether the comparison model receives an equivalent retraining budget. Avoid claiming a general cure from one trigger, one target and one dataset.

## Exercises and self-check

1. A report includes target-class examples in ASR. Recompute with 120 triggered examples, 20 target originals and 60 successful target predictions among non-target originals. **Check:** non-target ASR is 60%, not a denominator based on all 120.
2. Why must the clean and poisoned datasets be separate copies? **Check:** an in-place edit can poison the baseline or evaluation data and invalidate controls.
3. Design a table showing poison count, clean accuracy, clean-model trigger target rate, poisoned-model ASR and eligible count.
4. Explain why a low ASR for one trigger does not prove the model is backdoor-free. **Check:** the tested trigger family and attack process do not cover every possible mechanism.

## Primary reading

- [Gu, Dolan-Gavitt and Garg: BadNets](https://arxiv.org/abs/1708.06733): identify the supply-chain threat and compare its assumptions with this deliberately small local study.
- [NIST AI RMF 1.0](https://www.nist.gov/publications/artificial-intelligence-risk-management-framework-ai-rmf-10): connect the experiment to risk documentation and limitations rather than treating one score as a safety certificate.


# Evaluating defenses without a false sense of security

**Core time:** 120 minutes. **Prerequisites:** adversarial and backdoor experiments. **Outcomes:** compare clean utility and attack resistance; identify gradient masking symptoms; specify defender knowledge; design an adaptive, budget-aware evaluation plan.

## 1. A defense changes the system being evaluated

Input preprocessing, adversarial training, data filtering and randomized prediction modify different parts of a learning system. Write down the complete defended prediction function, including transforms and randomness. An attack against the old model alone may not evaluate the defended system. Conversely, an attack that ignores a transform's actual input scale can violate or underuse the intended threat budget.

For a local teaching comparison, consider a fixed smoothing transform before classification. Clean accuracy may drop because smoothing removes useful strokes. A weak attack may also become less effective. Neither observation alone establishes a useful defense. Measure both, use an attack suited to the actual defended computation and state what remains untested.

## 2. Robust optimization is a training objective

Adversarial training approximately minimizes a worst-case loss over a stated perturbation set. The inner maximization is implemented by a finite attack procedure and the outer minimization updates model parameters. This is computationally more expensive than a clean training step. Specify whether the loss uses clean and adversarial examples, how the inner attack is configured and whether gradients through the attack construction are retained or stopped.

```math
\min_\theta\;\mathbb{E}_{(x,y)}\left[\max_{\delta\in\Delta(x)}\ell(f_\theta(x+\delta),y)\right].
```

This expression defines the objective; the implementation only approximates it. A model trained against a particular norm budget may remain vulnerable to other shifts or perturbation families. A robustness gain at one epsilon is not a general privacy or fairness guarantee.

## 3. Worked comparison: calculate the actual tradeoff

Imagine a clean baseline is correct on 190 of 200 examples and remains correct on 80 under a tested attack suite. A defended model is clean-correct on 180 and remains correct on 110 under the same suite. Report clean accuracy 95% versus 90% and empirical suite survival 40% versus 55%. The defense sacrifices five points of clean accuracy for fifteen points of measured survival in this setting. Whether that trade is acceptable depends on the intended use.

Do not compare a baseline's 20-step attack with a defense's one-step attack or change the test subset to remove inconvenient failures. If reporting conditional attack success, denominators differ because clean-correct counts differ; show them explicitly alongside all-example metrics.



In [ ]:
rows = [{'name':'baseline','clean_correct':190,'survive':80},
        {'name':'defended','clean_correct':180,'survive':110}]
for row in rows:
    row['clean_accuracy'] = row['clean_correct']/200
    row['suite_survival'] = row['survive']/200
    assert row['survive'] <= row['clean_correct']
print(rows)



These counts are illustrative, not lab measurements. Your assignment table must come from saved predictions.

## 4. Gradient masking is an evaluation warning

A defense may make gradients discontinuous, stochastic or numerically unhelpful without removing vulnerable inputs. Warning patterns include iterative attacks unexpectedly underperforming simple attacks, large increases in attack budget producing no change, or black-box/transfer attacks finding failures missed by a naive white-box procedure. These are clues to investigate, not a theorem that any unusual curve proves masking.

Adaptive evaluation accounts for the defense's actual computation and randomness within the stated attacker knowledge. For stochastic systems, evaluate the relevant expected or probabilistic objective and report repetitions. For nondifferentiable transforms, an appropriate evaluation may require methods beyond the introductory gradient implementation. Do not claim a defense passes those tests if you have not run them.

## 5. Compare costs and information access

If the defense needs clean validation labels, a known trigger, additional training data or privileged model internals, include that requirement. Compare retraining budgets and report inference overhead. A detector's accuracy should include false alarms and rejection behavior; rejecting every input is not a useful classifier unless the task explicitly permits that policy and its consequences are reported.

## Exercises and self-check

1. A defense achieves high “accuracy on accepted inputs” while rejecting 90% of data. What is missing? **Check:** coverage, rejection rates by group/class, overall utility and the handling of rejected cases.
2. Why is a failure of one attack insufficient to certify robustness? **Check:** optimization may miss allowed adversarial points.
3. Design a defense comparison with matched selection data and declared extra computation. Include a no-defense control and clean evaluation.
4. A filtering defense removes 20% of training data. Propose a random-removal control and explain what it can and cannot isolate.

## Primary reading

- [Athalye, Carlini and Wagner: Obfuscated Gradients Give a False Sense of Security](https://arxiv.org/abs/1802.00420): study why an attack can fail for reasons other than robustness.
- [Madry et al.: adversarial robustness](https://arxiv.org/abs/1706.06083): connect the robust objective to the finite approximation used in practice.


# Privacy: membership risk, differential privacy and the meaning of a guarantee

**Core time:** 2 hours. **Prerequisites:** probability, expectations and training workflows. **Outcomes:** distinguish access controls from statistical privacy; define neighboring datasets; calculate sensitivity for a bounded statistic; explain composition and the ingredients of DP-SGD without inventing an epsilon guarantee.

## 1. Identify what an observer can learn

Keeping raw data on one machine reduces some exposure paths but does not prove that a released model reveals nothing about it. Model outputs, gradients, checkpoints and logs can carry information. A membership-inference question asks whether a particular record was included in training. Reconstruction asks a different question about recovering information. State the observer's access and auxiliary knowledge before evaluating either.

A weak empirical attack can fail even when information exists. Low membership-attack accuracy under one setup is therefore not a formal privacy guarantee. Conversely, a successful bounded test demonstrates a vulnerability under its assumptions, not a complete characterization of every privacy risk.

## 2. Differential privacy compares neighboring datasets

A randomized mechanism is approximately differentially private if its output probabilities are close for every pair of neighboring datasets and every output event, according to the definition below. Neighbors must be defined: add/remove one record, replace one record or add/remove one user's entire contribution are different choices.

```math
\Pr[M(D)\in S]\le e^\epsilon\Pr[M(D')\in S]+\delta.
```

Epsilon controls a multiplicative bound and delta an additive allowance. Smaller values generally correspond to a stronger guarantee under a fixed definition, but numbers from different adjacency units and accounting assumptions are not directly comparable. Delta is not simply “the probability one person's data are leaked.” The guarantee concerns distributions of outputs and a stated neighboring relation.

## 3. Worked example: a bounded mean

Assume exactly n=100 values in `[0,1]` and replace-one adjacency. Replacing one record changes their mean by at most 1/100, so its global L1 sensitivity is 0.01. The Laplace mechanism adds noise with scale `sensitivity/epsilon`; at epsilon 0.5 this is 0.02. The mathematical guarantee requires the stated bounds, adjacency and correctly sampled noise. If the data are not bounded, the sensitivity argument fails.



In [ ]:
import numpy as np
n,epsilon = 100,0.5
values = np.linspace(0,1,n)
sensitivity = 1/n
rng = np.random.default_rng(7)
noise = rng.laplace(0,sensitivity/epsilon,size=5000)
print({'true_mean_for_teaching':values.mean(),
       'noise_scale':sensitivity/epsilon,
       'noise_sample_std':noise.std(ddof=1)})



This code demonstrates a noise distribution on public synthetic values. It deliberately uses a reproducible seed for teaching and prints the true mean. It is **not a private release mechanism** for confidential data. Publishing the randomness or original sensitive answer can defeat the intended protection; production randomness, release control and accounting require an appropriate implementation.

## 4. Composition prevents unlimited free queries

Repeated releases consume privacy budget. Under basic sequential composition, releases with epsilons epsilon1 and epsilon2 can be bounded by their sum, with the corresponding delta sum for approximate DP. More advanced accountants can provide tighter bounds under their assumptions. Releasing many noisy answers and averaging them reduces noise and changes the privacy analysis; one query's epsilon cannot be reused as the claim for the whole history.

Post-processing a genuinely private output without additional sensitive access does not worsen that mechanism's guarantee. This does not mean that arbitrary preprocessing is private. Data-dependent clipping thresholds, model selection, hyperparameter tuning and extra statistics may themselves require accounting if they depend on protected data.

## 5. DP-SGD has several necessary parts

The usual conceptual recipe computes per-example gradients, clips each to a norm bound, aggregates them, adds calibrated noise and updates parameters. An accountant combines sampling assumptions, noise multiplier, clipping convention, number of steps and adjacency into a reported bound. Clipping the already averaged batch gradient is not the same operation as clipping each example. User-level privacy needs a user-level contribution definition, not merely a record-level accountant with a new label.

Our introductory lab checks the clipping arithmetic and demonstrates noise on synthetic data; it does not implement or advertise certified DP-SGD. For an extension, use a maintained privacy library, inspect its current documentation and report the accountant configuration. Never label a model “epsilon-private” because Gaussian noise was added somewhere in the loop.

## Exercises and self-check

1. For fixed n=50 values in `[0,10]` with replace-one adjacency, calculate mean sensitivity. **Check:** 10/50=0.2.
2. Two pure-DP releases use epsilons 0.3 and 0.7 on the same protected population. Give the basic composition bound. **Check:** epsilon 1.0, with the same adjacency assumptions.
3. Why does federated learning alone not imply DP? **Check:** locality is not a bound on how released updates/outputs change between neighboring datasets.
4. Explain why a fixed published random seed is inappropriate for claiming a confidential randomized release, even though it is useful for this public-data demonstration.

## Primary reading

- [Dwork and Roth: The Algorithmic Foundations of Differential Privacy](https://www.cis.upenn.edu/~aaroth/privacybook.html): consult definitions, sensitivity, Laplace noise and composition.
- [Abadi et al.: Deep Learning with Differential Privacy](https://arxiv.org/abs/1607.00133): identify per-example clipping, noise and privacy accounting; separate these ingredients from the toy illustration.


# Group evaluation, fairness criteria and calibrated claims

**Core time:** 120 minutes. **Prerequisites:** confusion matrices and calibration. **Outcomes:** compute group-conditioned rates with support counts; distinguish common fairness criteria; identify tradeoffs and missing context; avoid treating a toy metric as a social or legal verdict.

## 1. Start with the decision and affected population

A group disparity is a measurement requiring interpretation. Which groups matter, how membership is collected, what a label represents and what decisions follow are contextual questions. Labels may reflect historical decisions rather than an unbiased ground truth. An apparently equal metric can coexist with serious harms outside what was measured.

The course uses synthetic group labels to practice arithmetic. It does not infer demographic attributes or evaluate real people. A real assessment would require appropriate data governance, domain expertise and stakeholder input. The technical task here is to make metric definitions and limitations explicit.

## 2. Different criteria condition on different events

Demographic parity compares positive prediction rates across groups. Equality of opportunity compares true-positive rates among actual positives. Equalized odds compares both true-positive and false-positive rates. Calibration asks whether a given score corresponds to comparable observed outcome frequencies within a population/group. These criteria answer different questions and can conflict, especially when base rates differ and prediction is imperfect.

```math
\mathrm{TPR}_g=P(\hat Y=1\mid Y=1,G=g),\qquad
\mathrm{FPR}_g=P(\hat Y=1\mid Y=0,G=g).
```

Choosing one criterion is a substantive decision, not a library default. “Fairness improved” is incomplete without naming the criterion, groups, population, threshold and uncertainty. A smaller maximum gap may come from harming the better-performing group rather than improving outcomes for the worse-performing group; inspect absolute values and consequences.

## 3. Worked example: similar selection rates, unequal recall

Group A has 40 positive and 60 negative examples; the classifier finds 30 positives and falsely flags 10 negatives. Group B has 20 positive and 80 negative examples; it finds 10 positives and falsely flags 30 negatives. Both groups have 40 positive predictions out of 100, satisfying equal observed selection rates in this sample. Yet TPR is 0.75 versus 0.50 and FPR is about 0.167 versus 0.375. One parity statement clearly does not imply the others.



In [ ]:
groups = {'A':{'tp':30,'fn':10,'fp':10,'tn':50},
          'B':{'tp':10,'fn':10,'fp':30,'tn':50}}
def rates(c):
    total=sum(c.values())
    return {'count':total,'positive_count':c['tp']+c['fn'],
        'selection':(c['tp']+c['fp'])/total,
        'tpr':c['tp']/(c['tp']+c['fn']),
        'fpr':c['fp']/(c['fp']+c['tn'])}
result={g:rates(c) for g,c in groups.items()}
assert result['A']['selection'] == result['B']['selection']
print(result)



This example has nonzero denominators. In general, return an explicit missing/undefined value when a group has no positive or negative examples. Do not replace an undefined rate with zero and then claim the disparity is small.

## 4. Small groups require uncertainty, not silence

A rate based on five positives can change dramatically when one prediction changes. Report support counts, intervals or an appropriate uncertainty analysis. Group comparisons across many overlapping subgroups introduce selection concerns: discovering the worst gap after an unrestricted search is different from testing predeclared groups. Preserve the search scope and avoid selectively reporting only favorable slices.

Intersectional analysis can reveal patterns hidden in broad averages, but sparse intersections make estimates noisy. The response is not to erase small groups; state uncertainty and decide what additional evidence is needed. Bootstrap units must follow the independence structure, such as persons or devices rather than correlated rows.

## 5. Thresholds and mitigation change the policy

Changing a common threshold alters selection, recall and false-positive rates. Group-specific thresholds define a different decision policy and require contextual justification; they are not merely a numerical calibration trick. Fit any proposed policy on development data and evaluate it on held-out data. Show utility and group metrics together, along with coverage or abstention if used.

This lesson does not prescribe one universally correct fairness criterion. Its research standard is to explain the choice, expose the tradeoffs and avoid conclusions that exceed the evidence.

## Exercises and self-check

1. Compute precision for both groups in the worked example. **Check:** 30/40=0.75 and 10/40=0.25.
2. A group contains no true positives in the dataset. What is its empirical TPR? **Check:** undefined because the actual-positive denominator is zero.
3. Create a synthetic threshold sweep and plot absolute group rates plus their gap. Explain a case where a smaller gap accompanies worse performance for everyone.
4. Draft a model-card paragraph that reports one disparity without claiming the system is “fair” overall.

## Primary reading

- [Hardt, Price and Srebro: Equality of Opportunity in Supervised Learning](https://arxiv.org/abs/1610.02413): inspect conditional-rate definitions and post-processing assumptions.
- [Guo et al.: calibration](https://proceedings.mlr.press/v70/guo17a.html): distinguish probability calibration from thresholded error-rate parity.
- [Mitchell et al.: Model Cards](https://arxiv.org/abs/1810.03993): place subgroup evidence in a contextual report.


# Trustworthy-AI laboratory: a bounded audit with reproducible evidence

**Core time:** 3–4 hours. **Prerequisites:** Trustworthy AI 1–6. **Outcomes:** produce an auditable threat model, implement controls, report correctly conditioned metrics and write a model card with justified limitations.

## 1. Choose one primary investigation

The required laboratory contains a clean classifier, bounded adversarial evaluation and a controlled backdoor demonstration. For the assessed extension choose one primary investigation: perturbation-budget sensitivity, poisoning-fraction sensitivity or a clearly specified defense comparison. Privacy and group-metric demonstrations are conceptual checks, not claims that the digit model is formally private or socially fair.

Write the question before modifying code. For example: “At fixed model/data/training settings, how does a visible training trigger affect target predictions on non-target held-out digits?” Define the target, trigger, poisoning denominator, selection rule, eligible evaluation population and clean utility metric. A question this precise can produce an interpretable negative result.

## 2. Run and inspect the baseline artifacts

Run `python lab04_trust.py --output outputs/trust`. The output must distinguish clean-model metrics, attack constraints, conditioned success, poisoned-model clean utility and triggered target rate. Inspect the stored indices and counts. Verify that no training edit affected the validation or test arrays. Check epsilon-zero identity and legal input bounds before interpreting robustness numbers.

An output file saying “passed” is only as useful as its checks. Read which invariants are tested: projection around the original input, no trigger mutation of the original data, non-target ASR eligibility and finite gradients. State what remains untested, such as adaptive attacks against a new defense or broader trigger families.

## 3. A minimum evidence matrix

| Comparison | Fixed elements | Changed element | Required outcomes |
|---|---|---|---|
| Clean versus attacked input | Model, examples, labels | Permitted perturbation | Clean accuracy, attacked/suite survival, eligible count, actual norm |
| Clean versus poisoned training | Architecture, split, training policy | Training-only poison operation | Clean utility, target ASR, clean-model trigger control |
| Defense versus baseline | Threat model, evaluation population | Documented defense | Utility, resistance, knowledge assumptions and cost |

For each row, list the seeds and the precise budget. An apparent improvement caused by reducing attack steps or evaluating an easier subset is not evidence for the defense. A worse result under a stronger test may improve the audit by exposing a previously hidden weakness.

## 4. Worked interpretation of a null result

Suppose the poisoned model's triggered target rate is close to the clean model's rate. Do not conclude that backdoors are impossible. Check whether the trigger was actually inserted, whether selected labels were changed, whether the model fitted the training data and whether the eligible denominator is correct. If the implementation is valid, report that this particular poisoning rate, trigger, model and training budget did not produce a clear additional effect. That bounded result is scientifically useful.

If a stronger trigger or higher poison fraction is explored after seeing the result, label it as exploratory and use validation to develop the next protocol. Do not silently replace the original experiment with a favorable run while retaining a preregistration claim.

## 5. Model card and risk register

Your model card should state the training data and original attribution, intended educational use, excluded uses, preprocessing, architecture, selection, metrics, subgroup/shift scope, attack assumptions and unresolved limitations. The risk register should connect each observed issue to an action or a reason more evidence is needed. Avoid generic statements such as “all ethical risks considered.” Name the risks that were examined and those that were outside scope.

Provide representative failure examples with original labels and predictions. Visual illustrations support interpretation but do not replace aggregate counts. Do not choose only dramatic or successful examples; describe the sampling rule used to select them.

## Exercises and self-check

1. Write a one-page threat model and have a peer implement the evaluation without asking you which denominator to use. Revise any ambiguous field.
2. Intentionally include target originals in an ASR calculation, then show how the corrected eligibility changes the value. Explain why both numbers cannot share the same label.
3. Draft a defense claim that your experiment supports and a broader claim it does not support.
4. Explain why the privacy noise demonstration and fairness count example cannot be presented as deployment guarantees for the trained model.

## Submission readiness

Submit a 3–5 page report, runnable code, environment, split/poison indices, complete metric tables, representative examples, model card and risk register. The reviewer should be able to recover every reported denominator from saved counts. A high attack success rate is not the grading objective; correct threat-model reasoning and reproducible evidence are. Follow the module assignment's rubric and submit only your own local experiments.

## Primary reading

- [NIST AI RMF 1.0](https://www.nist.gov/publications/artificial-intelligence-risk-management-framework-ai-rmf-10): organize risks and actions.
- [BadNets](https://arxiv.org/abs/1708.06733): compare training-time assumptions.
- [Obfuscated Gradients](https://arxiv.org/abs/1802.00420): audit the limits of attack-based evidence.
